# Quantum Networking & Protocols

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThisIsBarney/quantx/blob/master/notebooks/04_quantum_networking.ipynb)  [![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/ThisIsBarney/quantx/master?filepath=notebooks/04_quantum_networking.ipynb)


This notebook demonstrates:
1. **BB84 key distribution** — QBER threshold and secure key rate
2. **Quantum teleportation** — fidelity formula $(2F+1)/3$ with Werner states
3. **Quantum network** — channel fidelity over distance


In [ ]:
# Install quantx-engine (needed in Colab / Binder)
# Skipped automatically if already installed in the local environment.
try:
    import quantx_engine  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                           'quantx-engine'])
    import quantx_engine  # noqa: F401
print('quantx_engine version:', quantx_engine.__version__)

## 1. BB84 Quantum Key Distribution

The Shor-Preskill secure key rate is $r = 1 - 2H_2(e)$, where $e$ is the QBER.
The threshold is at $e \approx 11\%$.  The `bb84_key_rate` function computes this analytically.

In [ ]:
from quantx_engine.quantum_protocols import BB84Protocol, bb84_key_rate

print('QBER   | Key rate (formula) | Secure?')
print('-' * 42)
for qber in [0.0, 0.05, 0.10, 0.11, 0.15, 0.20]:
    rate = bb84_key_rate(qber)
    secure = 'Yes' if rate > 0 else 'No '
    print(f'{qber:.2f}   |  {rate:+.4f}            | {secure}')

# Threshold check
assert bb84_key_rate(0.05) > 0, 'Should be secure at 5% QBER'
assert bb84_key_rate(0.20) <= 0, 'Should not be secure at 20% QBER'

In [ ]:
# Simulate BB84 protocol
result_lo = BB84Protocol(channel_error_rate=0.05, seed=42).run()
result_hi = BB84Protocol(channel_error_rate=0.20, seed=42).run()

print(f'BB84 (QBER=0.05): secure_key_rate={result_lo.secure_key_rate:.4f}, qber={result_lo.qber:.4f}')
print(f'BB84 (QBER=0.20): secure_key_rate={result_hi.secure_key_rate:.4f}, qber={result_hi.qber:.4f}')
assert result_lo.secure_key_rate > 0, 'Low-QBER protocol should yield positive key rate'
print('BB84 assertions passed.')

## 2. Quantum Teleportation

The teleportation average fidelity with a Werner state (entanglement fidelity $F$) is
$$\mathcal{F} = \frac{2F + 1}{3}$$
The `teleportation_fidelity` function implements this formula exactly.

In [ ]:
from quantx_engine.quantum_protocols import TeleportationProtocol, teleportation_fidelity

print('Werner F  |  Formula (2F+1)/3  |  Simulation avg_fidelity')
print('-' * 55)
for F_werner in [0.5, 0.7, 0.85, 1.0]:
    formula   = teleportation_fidelity(F_werner)
    sim       = TeleportationProtocol(channel_fidelity=F_werner, seed=0).run()
    print(f'F={F_werner:.2f}    |  {formula:.4f}              |  {sim.average_fidelity:.4f}')

# Perfect Bell pair should give teleportation fidelity 1
assert abs(teleportation_fidelity(1.0) - 1.0) < 1e-10, 'Perfect entanglement -> F_tele = 1'
# Classical limit: F=0.5 (maximally mixed) -> F_tele = 2/3
assert abs(teleportation_fidelity(0.5) - 2/3) < 1e-10, 'F=0.5 Werner -> F_tele = 2/3'
print('Teleportation fidelity formula verified.')

## 3. Quantum Network — Channel Fidelity over Distance

Entanglement fidelity decays exponentially with channel length due to attenuation.

In [ ]:
from quantx_engine.quantum_network import QuantumNetwork, QuantumChannel, QuantumNode

net = QuantumNetwork()
net.add_node(QuantumNode(node_id='Alice'))
net.add_node(QuantumNode(node_id='Bob'))
net.add_node(QuantumNode(node_id='Charlie'))

# Channels with different lengths
ch_ab = QuantumChannel.from_length('Alice', 'Bob',     length_km=10)
ch_bc = QuantumChannel.from_length('Bob',   'Charlie', length_km=50)
net.add_channel(ch_ab)
net.add_channel(ch_bc)

fmat = net.adjacency_fidelity()
f_ab = fmat[('Alice', 'Bob')]
f_bc = fmat[('Bob', 'Charlie')]
print(f'Alice-Bob  (10 km) Bell-pair fidelity: {f_ab:.4f}')
print(f'Bob-Charlie(50 km) Bell-pair fidelity: {f_bc:.4f}')

# Longer channel should have lower fidelity
assert f_ab > f_bc, 'Shorter channel should have higher fidelity'
assert f_ab > 0.5, 'Short channel should be above classical threshold'
print('Channel fidelity assertions passed.')

## Summary

| Module | Demonstrated |
|--------|--------------|
| `quantum_protocols.bb84_key_rate` | Shor-Preskill key rate formula, 11% QBER threshold |
| `quantum_protocols.BB84Protocol` | Simulated BB84 with realistic QBER |
| `quantum_protocols.teleportation_fidelity` | Werner state fidelity formula $(2F+1)/3$ |
| `quantum_network.QuantumNetwork` | Bell-pair fidelity vs. channel distance |

Next: **[05 — Many-Body & MPS](05_many_body_mps.ipynb)**